5.1回来，不知道哪根筋搭错了，突然对自动调参有了想法。一顿狂暴模式，花了4天写了个beta版出来，发布给算法用。又花了一周不断完善，想法越来越多，即使有code agent辅助，也还是需要斟酌各个想法的优先级。  
今天5.18，读了一篇ai新闻报道，说的是xAI收购cursor的事，指出100亿美金意在获取cursor的用户交互数据对模型做微调。这再次触发我新想法：这个自动调参agent，和code agent有什么区别呢？也是可以收集算法不断的调参尝试作为样本，对某个LLM模型进行微调。  
也许这想法来的有点晚：训个模型，是我从开始做agent相关，就不断有人在提的事。但我不能确定，提这些的人是否真的理解了这个模型该怎么训？反正我之前是不太理解的，现在也只是有了想法。  
anyway，沿着这个想法一路想，我觉得auto-tune-agent不是一个简单的玩具，它是一个需要长时间建设才可能看到效果的东西。我不确定身边的人及leader是否get到这点。目前有多个类似项目在进行，且目标都是要"做出业务效果"，也就是能通过agent提升某个实验的指标，以此来证明价值。以公司及领导的耐心，可能3个月就必须出结果。如果到时候没有效果，我不清楚这个项目是否还存在，是否能继续下去。而我的判断，agent出效果，还需要大量的工作。比如：eval体系建设——没有评估体系怎么定优化方向呢，context组织优化——如何组织context能获取更好结果，搜参方式——除了让模型做决策暴力搜之外还可以辅以离线搜参提升效率，更需要长期积累的：用户用的越多，调参数据就越多，这些数据也许可以微调一个模型，像for code的微调一样，提升调参效率，这个更是需要长期的数据积累。  
基于以上，要做一个真正有效的auto-tune-agent，可能需要几年的建设。如果在这里无法实现，那我只能换地方继续这个工作了。所以我需要把中间的思考记下来，方便未来迫不得已迁移。  
有意思的是，目前这个时代，已经没有必要"带着代码跑路"了。我只需要把思考的过程记下来，不带任何细节，就可以换一个地方重建，而且还方便因地制宜。所以我更得多记录了。

agent自动调参，后面统称为auto-tune-agent，和claude code/opencode/codex有什么区别？开始做agent应用相关时候，我以为所谓"agent xxx"就是基于某个code agent，如以上所述这些，将agent可以调用的接口通过tools/MCP/CLI等方式接入agent，然后再编写一些skill约束流程，然后再完善相关的知识库，即可。在这个架构里，code agent是核心入口，tools/MCP/CLI等是agent手脚，skill控制agent流程，整个流程入口就是聊天框输入。  
后来我发现，这样有很大局限性。入口是聊天框，也就是说交互模式就是“输入-输出”，一进一出。但想想auto-tune-agent这个场景，它的过程实际上是个loop，而不是"输入-输出"这种模式。auto-tune-agent的一个loop包含：LLM给建议->根据建议搜参->应用参数->等待ab结果->收集指标->评估结果指标和建议指标差异->归因/总结经验。之后是下一次循环，直到实验到达一定轮数或实验目标达成。在这个loop中，LLM的角色是其中一环——用来根据输入给建议，LLM不控制任何流程，全靠loop代码控制，LLM只负责在固定环节做决策。  
一旦你认识到，整个流程是个loop，而不是code agent中的“输入输出”模式，就发现，其实所有的所谓"agent"，都是个loop。包括claude code/opencode/codex，本质也是在LLM上包了一层代码做loop。这层代码针对coding环境，进行了一系列针对性的优化，如流程控制、session实现、context控制，等等，最终让LLM可以在coding这个场景下发挥作用。这么看，auto-tune-agent不应该只是code agent中的一个skill，而应该是一个平行关系的独立agent，有自己的loop循环、context控制、memory记录等等。也就是说：我们要建的是一个自动调参场景的claude code。  
再延伸一步，任何一个场景，想让LLM发挥作用，都需要一个代码来控制流程，LLM只是流程中负责某个决策的一环。这么看，有太多场景可以通过LLM去重塑了。

关于，自动调参的经验从哪来的，LLM怎么知道某个策略应该往哪调。  
开始我认为，调参经验来自于memory。比如某个在线参数调整，从memory中可以获取到，参数背景和历史调参之后的经验。这个经验既可以在memory中，通过召回的方式去拿到相关经验，也可以通过模型微调的方式，把经验内化到参数里。目前看比较灵活的是memory。  
也就是说，LLM构建了token之间的关系，理论上我们是把实验背景和历史调参经验输入进LLM，让LLM提取出其中相关的历史经验，然后作为prefix填入，让LLM推测下一步怎么调:  
LLM(background, round_history_1,round_history_2,round_history_3....)  
经过attention机制处理后等于:  
(background, round_history_1, round_history_40)  
相当于attention机制做了过滤。之后：  
LLM(background, round_history_1, round_history_40) -> next_action  
让LLM给出下一次参数调整。  
而第一步的过滤，有上下文限制，所以我们把这层移动到了调参框架里，也就是memory召回，在框架层先做一个硬过滤。  

然而，这样还是不够。因为不同调参点的经验是不同的。  
根本原因是，现在是拿(调参->ab指标)之间，做了个简单映射，LLM仅通过背景和ab指标，只能通过已经内化的知识来脑补中间的映射关系，然后去调。  
但实际上，从调参到用户指标，中间还有好几层映射：  
参数->排序结果->用户行为  
以及被调用户的属性，等等，都会影响最终的结果。  
但当前的自动调参全部跳过，直接用输入和输出做分析，其实能分析的有限。  
所以，还需要为agent补充离线分析的手段，去辅助判断。  

code vs llm  
我的设计是，整个loop，除了做参数建议部分，全都用代码实现。包括安全机制，如ab参数推全失败的回退等。我认为，只有在策略生成阶段需要LLM的探索能力，其他阶段都还是代码实现，来保证稳定性。  
今天看了另一个设计，是把整个loop切开，每一个step都嵌入LLM来决策下一步。比如，step A->LLM->step B->LLM->step C->LLM...。LLM在step A后获取到的是step A的执行结果，然后决策，下一步是step B还是C还是什么，对应的有一系列skill/tools来被调用。也就是说，由于每一个step都有LLM参与，形成了一个比较复杂的状态机，可以互相跳转。而我的loop是死的，就是一个loop循环，直到满足条件结束。  
这个设计和我的有什么本质区别吗？在我的设计中，LLM只是loop一环，给建议，无法决定下一步执行状态。而在另一个设计中，LLM真实决定了下一个状态。所以，我的方案更稳定、安全。而另一个方案更灵活，但不稳定，因此另一个方案也通过一些钩子等，在触发一些操作时进行检查等，来保证安全。  
实际上，灵活性和稳定性是两个极端。全部用代码实现，就是极致稳定性，没有任何“意外”，但也失去了探索能力。加入更多的LLM来决策，就增加了灵活性，也有了更多可能。缺点也就是可能走向歧途。在另一个方案中，通过一些机制来做兜底。  
这么看我觉得别人这个方案还真的挺好的。

其实，LLM的用法主要就是2个。一个是构建语言之间的联系，通过外部的memory/RAG等挂载额外的变化快的内容，通过LLM对token之间关系的构建，推理下一步，也就是续写。另外就是构建输入和tools调用之间的关系。也就是agent,给任务，输出tools调用链。我的自动调参loop把LLM约束到参数建议环节，其实是前者的用法，利用的是LLM构建的token间关系，核心是怎么构建输入触发合理的“续写”。而claude code等，以及RecAgent等，所有让agent决定调用什么工具的，都是后者用法，核心是给任务，输出tools调用链。

现在大多数工作其实都在工具接入agent上，解决的都是如何让agent来操作平台的问题。这个其实没什么技术含量，但确实有一些问题需要解决，也就是Agent的操作流程和人的操作流程必然不同。但这些跟最终的效果其实无关，跟agent设计的思路也无关，这些本身应该是平台去做的基建工作。现实问题是，除非公司推动所有平台agent化，不然平台没有一个合理的理由去做改造，需要业务去给效果产出，而业务就必须花不少力气去解决这个接入的问题。实际上从根本看，这个问题不决定效果、不重要。  
什么重要呢？agent的设计最重要，LLM到底在agent的loop里是个什么角色？不同思想设计出来的agent都不一样。具体哪个是有效的、哪个更好，目前没有答案，可能最终也不会有答案，看用户用哪个设计做出了什么效果。所以，各种agent必然会长期共存。而作为agent开发，需要关注的就是怎么设计这个agent能有效。

自动调参eval，应该也有一些封装好、标准的任务，交给自动调参agent去迭代，理论上应该用更少的轮数找到最优解。不过问题是，这种容易走着走着去优化解固定题目，但线上其实是开放世界任务，变量很多，很难说解固定题目的能力增强了，解线上问题就能提升。  
LLM的问题就是太难去评估这个效果到底好还是不好。  
或者仿照推荐系统ab的方式，直接放一部分流量出来，全权让agent去做优化，到季度末做pk，看谁涨的多？

策略反馈，人工备注对调参过程的影响太小，人工无法干预调参过程。背景是，自动调参出来的时长和VV都正向，而人其实要的是时长正向而VV可以负向，所以要求时长可以继续涨，涨到VV是负的且在合理范围内。  
分析了一下，根本原因是，
1. 实验配置了两个主指标，LLM自然会觉得两个都要提高，搞不清楚优先级，所以需要只配一个时长为主指标。
2. 指标约束只能配下限，不能配比例关系，比如vv约束应该是时长涨幅的-2倍
我开发支持了2，一个指标的约束可以依赖另一个指标。

另外，经过分析，我发现，其实调参是有整体策略的，比如初阶段先探索参数作用，然后再拉高主指标，逐步优化约束。这些是贯穿整个多轮调参的，当前的系统里没有地方指明。
所以我加了一个strategy的配置，可以写整体策略，每轮发给LLM按照策略调。

"认知更新"  
可以假设，调参过程是一个认知更新的过程。  
假设，有一个认知，如：这个参数对排序结果的影响是提升时长，拉低vv。这是一个认知。用公式表示，认知相当于一个映射过程中的参数w，输入是认知w，输出是线上指标结果。$y=f(w)$，这是极其简化的。  
我们要找的，是一个可以得到最优结果的"认知"，也就是w。  
你可以试10000次，获取到10000条结果，然后调整w，可以拟合这10000条样本，让$w=w^*$的情况下，样本产生的概率最高，这是最大化似然。  
也可以把w当成一个概率分布假设，作为先验，每做一次实验，就修正一次w，这是贝叶斯？  
ok，到这里有点乱七八糟，类比不太当。贝叶斯里w是个概率分布，有公式，有一系列假设，输入一条样本会输出一个分布，然后再从分布里挑一个置信度高的尝试，这是贝叶斯优化了。这里并没有什么分布，只有一个认知，或者经验。这样的话，似乎就是在找一个能满足最大化似然的$w^*$。  

没有code，提供tool并让模型做决策，不靠谱。  
我还是这个观点，LLM/Agent要做的事，必须用代码框架限制到足够小才行。整体流程一定是用代码实现，只有部分环节用agent解决人工定义好的问题，且要有严格校准机制。严禁agent自我发挥。  
现在某项目就遇到了一些问题。比如你提供了10个tool，但LLM选tool是概率性的，他可能一个tool都不选，而选择自己开始实现，这就导致没有按照预定路径执行，越走越远。好，就算选了tool，也可能发生意外错误、情况，这时候LLM又可能跑去改代码解决错误，进入另一条路径，等改完回来可能不记得自己要干什么了（当然，现在可以有状态保持机制让agent重新读干到哪了），总之，可能走入另一条未知的路径。这些情况都给稳定性带来了极大风险，LLM确实有可能解决未知问题，但也有可能把整个流程带入一个无人知道怎么处理的黑色地带。  
人们希望的是，LLM可以灵活解决一些问题，但同时又希望足够的稳定性，不要引发更多的问题。所以，LLM的决策权，一定要严格限制，让他的自我发挥在一个可控的范围内才行。